# OlpAI NLP: Fast Contest Transformer
Pipeline dịch Hoa → Việt đơn giản, không pretrained, tối ưu cho Colab T4. Notebook tự tải code + dữ liệu; chỉ cần chọn T4 rồi **Run all**. Mặc định chạy smoke test nhanh.

## 1. Cài thư viện

In [ ]:
%pip install -q sentencepiece sacrebleu tqdm pandas gdown

## 2. Tải code, dữ liệu và cấu hình
Cell này tự tải mọi thứ cần thiết. Lần đầu hãy giữ `SMOKE_TEST=True`; sau đó đổi thành `False` và chạy lại toàn bộ notebook để train bản thi thật.

In [ ]:
import os, shutil, subprocess, sys, time, zipfile
from pathlib import Path

# Repo chỉ chứa code. Dữ liệu được tải từ link Drive BTC đã cung cấp.
REPO_URL = 'https://github.com/TrDuy-pan3000/olpai-nlp-fast.git'
PROJECT_DIR = Path('/content/olpai-nlp-fast')
DATA_ROOT = Path('/content/olpai-data')
DATA_DIR = DATA_ROOT / 'dataset'
ARTIFACT_DIR = Path('/content/olpai-artifacts')

if not (PROJECT_DIR / 'nlp_basic.py').exists():
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(PROJECT_DIR)], check=True)
else:
    subprocess.run(['git', '-C', str(PROJECT_DIR), 'pull', '--ff-only'], check=True)

required_file = DATA_DIR / 'train/train.zh'
if not required_file.exists():
    import gdown
    DATA_ROOT.mkdir(parents=True, exist_ok=True)
    zip_path = DATA_ROOT / 'dataset.zip'
    gdown.download(id='190wIN301_X2Z7dqtPmLA7Z4Cggn3twl4', output=str(zip_path), quiet=False)
    with zipfile.ZipFile(zip_path) as archive:
        archive.extractall(DATA_ROOT)
assert required_file.exists(), f'Không tìm thấy dữ liệu tại {required_file}'

ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(PROJECT_DIR))

SMOKE_TEST = False # False = train thật toàn bộ dữ liệu; True chỉ để debug nhanh
USE_BEAM = False   # Luôn tạo greedy submission trước; beam chỉ bật khi còn thời gian
RESUME = False     # True: tiếp tục nếu last_model.pt vẫn còn trong runtime
SEED = 42

import torch
from nlp_basic import *

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)
if device.type == 'cuda':
    print('GPU:', torch.cuda.get_device_name(0))
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.set_float32_matmul_precision('high')
elif not SMOKE_TEST:
    raise RuntimeError('Fast Contest cần GPU T4. Chọn Runtime > Change runtime type > T4 GPU.')
seed_everything(SEED)
print('CHẾ ĐỘ:', 'SMOKE TEST (không có chất lượng)' if SMOKE_TEST else 'FULL CONTEST TRAINING')

## 3. Đọc, kiểm tra và chia dữ liệu

In [ ]:
TRAIN_ZH = DATA_DIR / 'train/train.zh'
TRAIN_VI = DATA_DIR / 'train/train.vi'
PUBLIC_ZH = DATA_DIR / 'public_test/public_test.zh'
PRIVATE_ZH = DATA_DIR / 'private_test/private_test.zh'

raw_pairs = read_parallel_data(TRAIN_ZH, TRAIN_VI)
memory = build_translation_memory([x[0] for x in raw_pairs], [x[1] for x in raw_pairs])
pairs = deduplicate_pairs(raw_pairs)
train_pairs, valid_pairs = group_split(pairs, valid_ratio=0.05, seed=SEED)
assert {s for s, _ in train_pairs}.isdisjoint({s for s, _ in valid_pairs})

if SMOKE_TEST:
    train_pairs = train_pairs[:1024]
    valid_pairs = valid_pairs[:256]

print(f'Raw: {len(raw_pairs):,} | Sau dedup: {len(pairs):,}')
print(f'Train: {len(train_pairs):,} | Valid: {len(valid_pairs):,}')
print('Ví dụ:', train_pairs[0])

## 4. Huấn luyện joint SentencePiece BPE

In [ ]:
VOCAB_SIZE = 2000 if SMOKE_TEST else 8000
tokenizer = train_sentencepiece(train_pairs, ARTIFACT_DIR, vocab_size=VOCAB_SIZE)
train_pairs = filter_pairs_by_token_length(train_pairs, tokenizer, max_len=40)
valid_pairs = filter_pairs_by_token_length(valid_pairs, tokenizer, max_len=40)
print(f'Sau lọc độ dài | Train: {len(train_pairs):,} | Valid: {len(valid_pairs):,}')
print('Vocab thực tế:', tokenizer.get_piece_size())
print('Token mẫu:', tokenizer.encode(train_pairs[0][0], out_type=str))

## 5. DataLoader và model

In [ ]:
if SMOKE_TEST:
    config = ModelConfig(vocab_size=tokenizer.get_piece_size(), d_model=96, nhead=4,
                         encoder_layers=1, decoder_layers=1, dim_feedforward=192,
                         dropout=0.1, max_len=32)
    BATCH_SIZE, EPOCHS, WARMUP, EVAL_EVERY = 64, 1, 20, 1
else:
    config = ModelConfig(vocab_size=tokenizer.get_piece_size(), d_model=256, nhead=8,
                         encoder_layers=4, decoder_layers=4, dim_feedforward=1024,
                         dropout=0.1, max_len=40)
    BATCH_SIZE, EPOCHS, WARMUP, EVAL_EVERY = 128, 15, 300, 2

train_loader = make_loader(train_pairs, tokenizer, BATCH_SIZE, config.max_len, True, 2)
valid_loader = make_loader(valid_pairs, tokenizer, BATCH_SIZE, config.max_len, False, 2)
model = Seq2SeqTransformer(config).to(device)
criterion = torch.nn.CrossEntropyLoss(ignore_index=PAD_ID, label_smoothing=0.1)
optimizer = make_optimizer(model, lr=5e-4)
scheduler = WarmupInverseSqrtScheduler(optimizer, peak_lr=5e-4, warmup_steps=WARMUP)
scaler = torch.amp.GradScaler('cuda', enabled=device.type == 'cuda')
print(f'Tham số: {sum(p.numel() for p in model.parameters()):,}')

## 6. Smoke check một batch

In [ ]:
src, tgt = next(iter(train_loader))
with torch.no_grad():
    logits = model(src.to(device), tgt[:, :-1].to(device))
assert logits.shape[:2] == tgt[:, 1:].shape
assert logits.shape[-1] == config.vocab_size
print('Smoke shape OK:', tuple(logits.shape))
del logits
if device.type == 'cuda': torch.cuda.empty_cache()

## 7. Huấn luyện
Fast Contest thường mất khoảng 8–18 phút train trên T4. BLEU chỉ chạy mỗi 2 epoch để tiết kiệm thời gian.

In [ ]:
best_bleu, bad_evals, start_epoch = -1.0, 0, 1
last_path = ARTIFACT_DIR / 'last_model.pt'
if RESUME and last_path.exists():
    state = load_checkpoint(last_path, model, optimizer, scheduler, scaler, device)
    start_epoch = state['epoch'] + 1
    best_bleu = state['best_bleu']
    print(f'Resume từ epoch {state["epoch"]}, best BLEU={best_bleu:.2f}')
start_time = time.time()
for epoch in range(start_epoch, EPOCHS + 1):
    train_loss = train_one_epoch(model, train_loader, optimizer, scheduler, criterion,
                                 device, scaler, epoch)
    val_loss = evaluate_loss(model, valid_loader, criterion, device)
    should_eval = (epoch % EVAL_EVERY == 0) or epoch == EPOCHS or SMOKE_TEST
    bleu = evaluate_bleu(model, valid_loader, tokenizer, device, config.max_len) if should_eval else None

    save_checkpoint(ARTIFACT_DIR / 'last_model.pt', model, optimizer, scheduler,
                    epoch, best_bleu, config, scaler)
    if bleu is not None:
        if bleu > best_bleu:
            best_bleu, bad_evals = bleu, 0
            save_checkpoint(ARTIFACT_DIR / 'best_model.pt', model, optimizer, scheduler,
                            epoch, best_bleu, config, scaler)
        else:
            bad_evals += 1
    print(f'Epoch {epoch:02d} | train={train_loss:.3f} | val={val_loss:.3f} | '
          f'BLEU={bleu if bleu is not None else "skip"} | best={best_bleu:.2f}')
    if bad_evals >= 3:
        print('Early stopping: BLEU không tăng sau 3 lần đánh giá.')
        break
print(f'Tổng thời gian train: {(time.time()-start_time)/60:.1f} phút')

## 8. Xem nhanh một số bản dịch validation

In [ ]:
load_checkpoint(ARTIFACT_DIR / 'best_model.pt', model, device=device)
sample_src = [s for s, _ in valid_pairs[:10]]
sample_ref = [t for _, t in valid_pairs[:10]]
sample_pred = translate_sentences(model, sample_src, tokenizer, device, batch_size=10, max_len=config.max_len)
import pandas as pd
display(pd.DataFrame({'Hoa': sample_src, 'Tham chiếu': sample_ref, 'Mô hình': sample_pred}))

## 9. Tạo submission
Greedy theo batch là mặc định nhanh. Chỉ bật `USE_BEAM=True` nếu đã có submission greedy an toàn và còn thời gian.

In [ ]:
from tqdm.auto import tqdm
public_src, private_src = read_lines(PUBLIC_ZH), read_lines(PRIVATE_ZH)

def make_predictions(sentences):
    if not USE_BEAM:
        return translate_sentences(model, sentences, tokenizer, device, memory,
                                   batch_size=BATCH_SIZE, max_len=config.max_len)
    output = []
    for sentence in tqdm(sentences, desc='Beam-3 inference'):
        key = normalize_text(sentence)
        output.append(memory[key] if key in memory else
                      beam_search_decode_sentence(model, key, tokenizer, device, 3, config.max_len, 0.6))
    return output

public_pred = make_predictions(public_src)
private_pred = make_predictions(private_src)
public_csv = write_submission_csv(public_src, public_pred, ARTIFACT_DIR / 'public_test.csv')
private_csv = write_submission_csv(private_src, private_pred, ARTIFACT_DIR / 'private_test.csv')
submission = package_submission(public_csv, private_csv, ARTIFACT_DIR / 'submission.zip')
print('Đã tạo:', submission)

# Colab tự bật hộp tải file kết quả về máy.
from google.colab import files
files.download(str(submission))

## 10. Chiến thuật phòng thi
1. Chạy smoke test. 2. Chuyển `SMOKE_TEST=False`, chọn T4 và Run all. 3. Tạo greedy submission ngay. 4. Chỉ thử beam-3 hoặc tăng epoch nếu còn thời gian. Hãy tải `submission.zip` và checkpoint về máy trước khi đóng runtime.